In [27]:
import pandas as pd

df = pd.read_csv("../data/processed_text.csv", keep_default_na=False)


In [28]:
from sklearn.model_selection import train_test_split

X = df["processed_text"] # clean, lemmatized text
y = df[["toxic","severe_toxic","obscene","threat","insult","identity_hate"]] # labels

# Train and test with an 80-20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.head()


140030    grandma terri burn trash grandma terri trash h...
159124    utc easiest admit member involve portuguese lo...
60006     objectivity discussion doubtful nonexistent in...
65432                             shelly shock shelly shock
154979    care refer ong teng cheong talk page la goutte...
Name: processed_text, dtype: object

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import joblib
import numpy as np

# Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=50000,   # keeps it manageable but still powerful
    ngram_range=(1,2),    # unigrams + bigrams
)

# Fit on training text, transform both train and test
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Quick check of shape
X_train_vec.shape


(127656, 50000)

In [37]:
joblib.dump(vectorizer, "../data/tfidf.pkl")
sparse.save_npz("../data/X_train_tfidf.npz", X_train_vec)
sparse.save_npz("../data/X_test_tfidf.npz", X_test_vec)
np.save("../data/y_train.npy", y_train)
np.save("../data/y_test.npy", y_test)
np.save("../data/label_names.npy", y_train.columns)


# Calibrated LinearSVC (CalibratedClassifierCV wrap) Attempts

In [17]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier

base_svm = LinearSVC()

calibrated_svm = OneVsRestClassifier(
    CalibratedClassifierCV(base_svm, cv=3, method="sigmoid")
)

calibrated_svm.fit(X_train_vec, y_train)

y_pred_cal = calibrated_svm.predict(X_test_vec)

from sklearn.metrics import classification_report, f1_score

print("Macro F1-score:", f1_score(y_test, y_pred_cal, average="macro"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_cal, target_names=y_train.columns))


Macro F1-score: 0.5247190976545147

Classification Report:

               precision    recall  f1-score   support

        toxic       0.88      0.66      0.76      3056
 severe_toxic       0.54      0.20      0.29       321
      obscene       0.90      0.69      0.78      1715
       threat       0.55      0.24      0.34        74
       insult       0.81      0.53      0.64      1614
identity_hate       0.70      0.23      0.35       294

    micro avg       0.86      0.60      0.70      7074
    macro avg       0.73      0.43      0.52      7074
 weighted avg       0.84      0.60      0.69      7074
  samples avg       0.06      0.05      0.05      7074



c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

# Linear SVM Attemps

In [15]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

svm_model = OneVsRestClassifier(
    LinearSVC()
)

svm_model.fit(X_train_vec, y_train)


,estimator,LinearSVC()
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [16]:
from sklearn.metrics import classification_report, f1_score

y_pred_svm = svm_model.predict(X_test_vec)

print("Macro F1-score:", f1_score(y_test, y_pred_svm, average="macro"))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm, target_names=y_train.columns))

Macro F1-score: 0.5271388561187131

Classification Report:

               precision    recall  f1-score   support

        toxic       0.86      0.69      0.76      3056
 severe_toxic       0.49      0.22      0.31       321
      obscene       0.88      0.70      0.78      1715
       threat       0.47      0.20      0.28        74
       insult       0.79      0.57      0.66      1614
identity_hate       0.70      0.25      0.37       294

    micro avg       0.83      0.62      0.71      7074
    macro avg       0.70      0.44      0.53      7074
 weighted avg       0.82      0.62      0.70      7074
  samples avg       0.06      0.06      0.06      7074



c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

# Initial Logistic Regression Attempt

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report

# One-vs-Rest Logistic Regression
lr = LogisticRegression(
    solver="saga",        # good for large sparse datasets
    max_iter=1000,        # ensure convergence
    n_jobs=-1             # use all CPU cores
)

model = OneVsRestClassifier(lr)

# Train the model
model.fit(X_train_vec, y_train)

# Make predictions
y_pred = model.predict(X_test_vec)

# Evaluate with Macro F1-score
f1_macro = f1_score(y_test, y_pred, average='macro')
print("Macro F1-score:", f1_macro)

# Detailed per-label report
print(classification_report(y_test, y_pred, target_names=y.columns))


Macro F1-score: 0.47325739663637134
               precision    recall  f1-score   support

        toxic       0.92      0.60      0.72      3056
 severe_toxic       0.54      0.21      0.30       321
      obscene       0.92      0.62      0.74      1715
       threat       0.56      0.12      0.20        74
       insult       0.81      0.49      0.61      1614
identity_hate       0.80      0.16      0.27       294

    micro avg       0.88      0.54      0.67      7074
    macro avg       0.76      0.37      0.47      7074
 weighted avg       0.87      0.54      0.66      7074
  samples avg       0.05      0.05      0.05      7074



c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 